# Deep Learning Assignments 1-6 (Ubuntu + Jupyter, Python 3)
**One-time setup in the Ubuntu terminal:**
```bash
sudo apt update && sudo apt install -y python3-pip python3-venv
python3 -m venv ~/dl-env && source ~/dl-env/bin/activate
pip install notebook numpy pandas matplotlib scikit-learn tensorflow
# only for Assignment 1 (optional): pip install torch --index-url https://download.pytorch.org/whl/cpu
jupyter notebook
```
Datasets (MNIST, CIFAR-10) download automatically on first run - no local `.npz` files needed.

## Assignment 1 - Check library versions

In [ ]:
import numpy as np
print("NumPy     :", np.__version__)

import tensorflow as tf
print("TensorFlow:", tf.__version__)

import keras
print("Keras     :", keras.__version__)

try:
    import torch
    print("PyTorch   :", torch.__version__)
except ImportError:
    print("PyTorch not installed (optional): pip install torch --index-url https://download.pytorch.org/whl/cpu")

## Assignment 2 - Feed-forward network on MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.datasets import mnist

(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
X_train = X_train.reshape((X_train.shape[0], 28 * 28)).astype("float32") / 255.0
X_test  = X_test.reshape((X_test.shape[0], 28 * 28)).astype("float32") / 255.0

lb = LabelBinarizer()
Y_train = lb.fit_transform(Y_train)
Y_test  = lb.transform(Y_test)

In [ ]:
model = Sequential([
    Input(shape=(784,)),
    Dense(128, activation="sigmoid"),
    Dense(64,  activation="sigmoid"),
    Dense(10,  activation="softmax"),
])
epochs = 10
model.compile(loss="categorical_crossentropy", optimizer=SGD(0.01), metrics=["accuracy"])
H = model.fit(X_train, Y_train, validation_data=(X_test, Y_test),
              epochs=epochs, batch_size=128)

In [ ]:
predictions = model.predict(X_test, batch_size=128)
print(classification_report(Y_test.argmax(axis=1),
                            predictions.argmax(axis=1),
                            target_names=[str(c) for c in lb.classes_]))

In [ ]:
plt.style.use("ggplot")
plt.figure()
plt.plot(np.arange(0, epochs), H.history["loss"],         label="train_loss")
plt.plot(np.arange(0, epochs), H.history["val_loss"],     label="val_loss")
plt.plot(np.arange(0, epochs), H.history["accuracy"],     label="train_acc")
plt.plot(np.arange(0, epochs), H.history["val_accuracy"], label="val_acc")
plt.title("Training Loss and Accuracy")
plt.xlabel("Epoch #"); plt.ylabel("Loss/Accuracy"); plt.legend()
plt.show()

## Assignment 3A - CNN on MNIST
Fix: the original used plain `sgd` with default settings and stayed at ~11% accuracy (not learning). Using `adam` fixes it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Input
from tensorflow.keras.datasets import mnist

(xtrain, ytrain), (xtest, ytest) = mnist.load_data()
xtrain = xtrain.reshape((-1, 28, 28, 1)).astype("float32") / 255.0
xtest  = xtest.reshape((-1, 28, 28, 1)).astype("float32") / 255.0

plt.figure(figsize=(5, 5))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(xtrain[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.show()

In [ ]:
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(100, activation="relu"),
    Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(xtrain, ytrain, epochs=5, validation_data=(xtest, ytest))

In [ ]:
import random
predictions = model.predict(xtest)
index = random.randint(0, len(xtest) - 1)
plt.imshow(xtest[index].squeeze(), cmap="gray")
plt.title(f"Predicted: {predictions[index].argmax()}   Actual: {ytest[index]}")
plt.show()

print("Test loss, accuracy:", model.evaluate(xtest, ytest, verbose=0))

In [ ]:
plt.figure(figsize=(13, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epochs"); plt.ylabel("loss"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Testing accuracy")
plt.xlabel("Epochs"); plt.ylabel("accuracy"); plt.legend()
plt.show()

## Assignment 3B - CNN on CIFAR-10
Fixes: dataset loaded directly from Keras, images normalised, plot grid fixed (2x5), one clean training run instead of two.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images = train_images.astype("float32") / 255.0
test_images  = test_images.astype("float32") / 255.0
print("Training data:", train_images.shape)
print("Testing data :", test_images.shape)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10),
])
model.summary()

In [ ]:
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
h = model.fit(train_images, train_labels, epochs=10,
              validation_data=(test_images, test_labels))

In [ ]:
plt.plot(h.history["accuracy"], label="accuracy")
plt.plot(h.history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
plt.show()
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print("Test accuracy:", test_acc)

## Assignment 4 - Autoencoder for ECG anomaly detection
Fixes: `https` URL, `Input` layer, `msle` argument order (y_true, y_pred), errors converted to NumPy before comparing with the threshold. Label 1 = normal ECG, 0 = anomaly.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout

PATH_TO_DATA = "https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv"
data = pd.read_csv(PATH_TO_DATA, header=None)
print(data.shape)
data.head()

In [ ]:
features = data.drop(140, axis=1)
target = data[140]
x_train, x_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=42)

# train only on normal ECGs (label == 1)
train_data = x_train.loc[y_train[y_train == 1].index]

scaler = MinMaxScaler(feature_range=(0, 1))
x_train_scaled = scaler.fit_transform(train_data.copy())
x_test_scaled  = scaler.transform(x_test.copy())

In [ ]:
class AutoEncoder(Model):
    def __init__(self, output_units, ldim=8):
        super().__init__()
        self.encoder = Sequential([
            Dense(64, activation="relu"), Dropout(0.1),
            Dense(32, activation="relu"), Dropout(0.1),
            Dense(16, activation="relu"), Dropout(0.1),
            Dense(ldim, activation="relu"),
        ])
        self.decoder = Sequential([
            Dense(16, activation="relu"), Dropout(0.1),
            Dense(32, activation="relu"), Dropout(0.1),
            Dense(64, activation="relu"), Dropout(0.1),
            Dense(output_units, activation="sigmoid"),
        ])

    def call(self, inputs):
        return self.decoder(self.encoder(inputs))

model = AutoEncoder(output_units=x_train_scaled.shape[1])
model.compile(loss="msle", metrics=["mse"], optimizer="adam")
history = model.fit(x_train_scaled, x_train_scaled, epochs=20, batch_size=512,
                    validation_data=(x_test_scaled, x_test_scaled))

In [ ]:
plt.plot(history.history["loss"], label="loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epochs"); plt.ylabel("MSLE Loss"); plt.legend()
plt.show()

In [ ]:
def find_threshold(model, x_train_scaled):
    recon = model.predict(x_train_scaled)
    errors = tf.keras.losses.msle(x_train_scaled, recon).numpy()
    return errors.mean() + errors.std()

def get_predictions(model, x_test_scaled, threshold):
    recon = model.predict(x_test_scaled)
    errors = tf.keras.losses.msle(x_test_scaled, recon).numpy()
    # error above threshold -> anomaly (0), else normal (1)
    return np.where(errors > threshold, 0.0, 1.0)

threshold = find_threshold(model, x_train_scaled)
print("Threshold:", threshold)

predictions = get_predictions(model, x_test_scaled, threshold)
print("Accuracy:", accuracy_score(y_test, predictions))

## Assignment 5 - CBOW (Word2Vec) from scratch with NumPy
Fixes: completed the truncated `range(...)` line in `generate_training_data`.

In [ ]:
import re
import numpy as np
np.random.seed(42)

corpus = """
The quick brown fox jumps over the lazy dog.
The dog barks at the fox and the fox runs away.
A quick brown dog jumps over the lazy fox.
Natural language processing helps computers understand human language.
Word embeddings capture the meaning of words in vectors.
The model learns word vectors by predicting a word from its context.
Computers learn language from large amounts of text.
The cat and the dog are friends and the cat likes the dog.
"""

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

tokens = preprocess(corpus)
vocab = sorted(set(tokens))
V = len(vocab)
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("Total tokens:", len(tokens))
print("Vocab size  :", V)
print("Vocabulary  :", vocab)

In [ ]:
WINDOW = 2

def generate_training_data(tokens, window):
    data = []
    for i, target in enumerate(tokens):
        context = [tokens[j]
                   for j in range(max(0, i - window), min(len(tokens), i + window + 1))
                   if j != i]
        data.append((context, target))
    return data

def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1
    return v

pairs = generate_training_data(tokens, WINDOW)
X = [[word2idx[w] for w in ctx] for ctx, _ in pairs]
y = [word2idx[t] for _, t in pairs]

for ctx, tgt in pairs[:5]:
    print(f"context={ctx} -> target='{tgt}'")
print("Total training samples:", len(pairs))

In [ ]:
N = 10          # embedding dimension
EPOCHS = 500
LR = 0.05

W1 = np.random.randn(V, N) * 0.1
W2 = np.random.randn(N, V) * 0.1

def softmax(u):
    e = np.exp(u - np.max(u))
    return e / e.sum()

def forward(context_idx):
    x_avg = np.mean([one_hot(i, V) for i in context_idx], axis=0)
    h = x_avg @ W1
    u = h @ W2
    y_hat = softmax(u)
    return x_avg, h, y_hat

loss_history = []
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for ctx_idx, tgt_idx in zip(X, y):
        x_avg, h, y_hat = forward(ctx_idx)
        total_loss += -np.log(y_hat[tgt_idx] + 1e-9)

        e = y_hat.copy()
        e[tgt_idx] -= 1
        dW2 = np.outer(h, e)
        dh  = W2 @ e
        dW1 = np.outer(x_avg, dh)

        W2 -= LR * dW2
        W1 -= LR * dW1

    loss_history.append(total_loss / len(X))
    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}/{EPOCHS}  loss = {loss_history[-1]:.4f}")

In [ ]:
def predict(context_words, top_k=3):
    idxs = [word2idx[w] for w in context_words]
    _, _, y_hat = forward(idxs)
    top = np.argsort(y_hat)[::-1][:top_k]
    return [(idx2word[i], round(float(y_hat[i]), 4)) for i in top]

def most_similar(word, top_k=3):
    emb = W1 / (np.linalg.norm(W1, axis=1, keepdims=True) + 1e-9)
    sims = emb @ emb[word2idx[word]]
    best = np.argsort(sims)[::-1][1:top_k + 1]
    return [(idx2word[i], round(float(sims[i]), 3)) for i in best]

tests = [["the", "quick", "fox", "jumps"],
         ["the", "lazy", "the", "barks"],
         ["word", "capture", "meaning", "of"]]
for ctx in tests:
    print(f"Context {ctx} -> predicted: {predict(ctx)}")

print("\nEmbedding of 'fox':")
print(np.round(W1[word2idx['fox']], 3))

print("\nMost similar words:")
for w in ["fox", "language", "dog"]:
    print(f"  {w}: {most_similar(w)}")

print(f"\nFinal loss: {loss_history[-1]:.4f} (started at {loss_history[0]:.4f})")

## Assignment 6 - Transfer learning (VGG16 + CIFAR-10)
Fixes: `numpy` import added, CIFAR-10 loaded from Keras, truncated `metrics=['accuracy']` completed, VGG16's own `preprocess_input` used instead of `/255`, `Input` layer added. On CPU this is slow (~5 min/epoch); reduce `epochs` if needed.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.utils import to_categorical

(itrain, ltrain), (itest, ltest) = cifar10.load_data()

itrain = preprocess_input(itrain.astype("float32"))
itest  = preprocess_input(itest.astype("float32"))

ltrain = to_categorical(ltrain, 10)
ltest  = to_categorical(ltest, 10)
print(itrain.shape, ltrain.shape)

In [ ]:
basem = VGG16(weights="imagenet", include_top=False, input_shape=(32, 32, 3))
for layer in basem.layers:
    layer.trainable = False

semodel = Sequential([
    Input(shape=(32, 32, 3)),
    basem,
    Flatten(),
    Dense(256, activation="relu"),
    Dense(10, activation="softmax"),
])
semodel.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
semodel.summary()

In [ ]:
history = semodel.fit(itrain, ltrain, epochs=10, batch_size=64,
                      validation_data=(itest, ltest))

In [ ]:
loss, acc = semodel.evaluate(itest, ltest)
print("\nTest accuracy:", acc)